# DWT-Enhanced FPN for Microscopic Nuclei Segmentation (MoNuSeg)
Self-contained Kaggle notebook version of the `code/FPN` scripts.

**Before running:** Session options → Accelerator = **GPU (T4/P100)**, Internet = **On** (for ImageNet weights).

Dataset: `tuanledinh/monuseg2018` (images are `.tif`, masks are `.png` with a trailing space in the name — handled below).

In [ ]:
# ============ 0. Environment check ============
import os, time, csv
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, jaccard_score,
                             precision_score, recall_score, roc_auc_score)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============ 1. Configuration (replaces argparse) ============
class Config:
    # Data (Kaggle paths)
    base = "/kaggle/input/datasets/tuanledinh/monuseg2018"
    train_img_dir  = f"{base}/kmms_training/kmms_training/images"
    train_mask_dir = f"{base}/kmms_training/kmms_training/masks"
    test_img_dir   = f"{base}/kmms_test/kmms_test/images"
    test_mask_dir  = f"{base}/kmms_test/kmms_test/masks"

    # Model
    backbone = "resnet50"
    pretrained = True            # ImageNet init
    pretrained_weights = None    # path to UNITPathSSL checkpoint (upload as Kaggle dataset), or None
    num_classes = 1

    # Training (thesis hyperparameters, adapted for Kaggle GPU: real batching instead of accumulation)
    lr = 5e-5
    weight_decay = 1e-4
    epochs = 30
    batch_size = 4
    accumulation_steps = 1
    target_size = (512, 512)
    freeze_bn = True             # was required for batch-size-1 on 4GB GPU; try False with batch>=4

    # Augmentation
    augment = True
    augment_prob = 0.6
    augment_factor = 3
    rotation_range = (-30, 30)
    scale_range = (0.85, 1.15)

    # Runtime
    num_workers = 2
    save_dir = "/kaggle/working/checkpoints/final"

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
assert os.path.isdir(cfg.train_img_dir), f"Image dir not found: {cfg.train_img_dir}"
assert os.path.isdir(cfg.train_mask_dir), f"Mask dir not found: {cfg.train_mask_dir}"
print("Train images:", len(os.listdir(cfg.train_img_dir)))

In [ ]:
# ============ 2. Dataset (from dataset.py) ============
def list_image_files(img_dir, extensions=None):
    if extensions is None:
        extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
    return sorted([f for f in os.listdir(img_dir) if f.lower().endswith(extensions)])


def split_image_list(image_files, test_size=0.1, val_size=0.1, random_state=None):
    if val_size < 0 or test_size < 0 or val_size + test_size >= 1.0:
        raise ValueError("val_size and test_size must be non-negative and sum to less than 1.0")
    train_val, test_files = train_test_split(
        image_files, test_size=test_size, random_state=random_state, shuffle=True)
    if val_size == 0.0:
        return train_val, [], test_files
    val_fraction = val_size / (1.0 - test_size)
    train_files, val_files = train_test_split(
        train_val, test_size=val_fraction, random_state=random_state, shuffle=True)
    return train_files, val_files, test_files


class MoNuSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, target_size=(512, 512), augment=False,
                 augment_factor=1, augment_prob=0.5, rotation_range=(-15, 15),
                 scale_range=(0.9, 1.1), image_list=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.target_size = tuple(target_size)
        self.augment = augment
        self.augment_factor = augment_factor
        self.augment_probability = augment_prob
        self.rotation_range = rotation_range
        self.scale_range = scale_range

        valid_extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
        if image_list is None:
            self.images = list_image_files(img_dir, extensions=valid_extensions)
        else:
            self.images = [f for f in image_list if f.lower().endswith(valid_extensions)]

        self.samples = []
        for img_name in self.images:
            self.samples.append({"img_name": img_name, "augment": False})
            if self.augment:
                for _ in range(self.augment_factor):
                    self.samples.append({"img_name": img_name, "augment": True})
        if self.augment:
            np.random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def _augment_pair(self, img, mask):
        angle = np.random.uniform(self.rotation_range[0], self.rotation_range[1])
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        h, w = img.shape[:2]
        center = (w / 2.0, h / 2.0)

        # Zoom in enough so the rotated image still covers the full canvas
        # (no border extrapolation needed; avoids mirror seams / black wedges)
        rad = np.deg2rad(abs(angle))
        cover_scale = np.cos(rad) + np.sin(rad)
        scale = max(scale, cover_scale)

        matrix = cv2.getRotationMatrix2D(center, angle, scale)
        img = cv2.warpAffine(img, matrix, (w, h), flags=cv2.INTER_LINEAR)
        mask = cv2.warpAffine(mask, matrix, (w, h), flags=cv2.INTER_NEAREST)
        return img, mask

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_name = sample["img_name"]
        should_augment = sample["augment"]

        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        base_name = os.path.splitext(img_name)[0]
        mask_candidates = (
            f"{base_name}_bin_mask.png",
            f"{base_name}.png",
            f"{base_name} .png",  # tuanledinh/monuseg2018 masks have a trailing space
        )
        mask = None
        for mask_name in mask_candidates:
            mask_path = os.path.join(self.mask_dir, mask_name)
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, 0)
                if mask is not None:
                    break
        if mask is None:
            raise FileNotFoundError(f"Could not find mask for {img_name} at {self.mask_dir}")

        img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        if should_augment and (np.random.rand() < self.augment_probability):
            img, mask = self._augment_pair(img, mask)

        img = np.ascontiguousarray(img)
        mask = np.ascontiguousarray(mask)

        mask = (mask > 0).astype("float32")
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        return img_tensor, mask_tensor

In [ ]:
# ============ 3. Loss (from loss.py) ============
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1.0 - ((2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth))

def total_loss(pred, target):
    """Downstream Supervised Fine-Tuning Loss."""
    return bce_loss(pred, target) + dice_loss(pred, target)

In [ ]:
# ============ 4. DWT Entropy Fusion (from DWTFusion.py) ============
class DWTEntropyFusion(nn.Module):
    """Entropy-guided spatial attention fusion for Wavelet high-frequency components."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_reduce = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.attn_conv = nn.Sequential(
            nn.Conv2d(in_channels * 3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=1)
        )

    def local_entropy(self, x):
        p = torch.abs(x)
        p = F.avg_pool2d(p, kernel_size=3, stride=1, padding=1)
        p = p / (p.sum(dim=(2, 3), keepdim=True) + 1e-8)
        ent = -p * torch.log(p + 1e-8)
        return torch.mean(ent, dim=1, keepdim=True)

    def forward(self, LH, HL, HH):
        LH = (LH - LH.mean(dim=(2, 3), keepdim=True)) / (LH.std(dim=(2, 3), keepdim=True) + 1e-6)
        HL = (HL - HL.mean(dim=(2, 3), keepdim=True)) / (HL.std(dim=(2, 3), keepdim=True) + 1e-6)
        HH = (HH - HH.mean(dim=(2, 3), keepdim=True)) / (HH.std(dim=(2, 3), keepdim=True) + 1e-6)

        e_lh = self.local_entropy(LH)
        e_hl = self.local_entropy(HL)
        e_hh = self.local_entropy(HH)
        entropy_weights = torch.cat([e_lh, e_hl, e_hh], dim=1)

        feat = torch.cat([LH, HL, HH], dim=1)
        attn = self.attn_conv(feat)

        weights = F.softmax(entropy_weights + attn, dim=1)
        F_out = weights[:, 0:1] * LH + weights[:, 1:2] * HL + weights[:, 2:3] * HH
        return self.conv_reduce(F_out)

In [ ]:
# ============ 5. Modified FPN (from ModifiedFPN.py) ============
class ModifiedFPN(nn.Module):
    """FPN with Group Normalization (32 groups), DWT Entropy Fusion, and P6 level."""
    def __init__(self, in_channels=(256, 512, 1024, 2048), out_channels=256, num_groups=32):
        super().__init__()
        if len(in_channels) != 4:
            raise ValueError("ModifiedFPN requires exactly 4 backbone feature channels")

        self.lat2 = nn.Sequential(nn.Conv2d(in_channels[0], out_channels, kernel_size=1),
                                  nn.GroupNorm(num_groups, out_channels))
        self.lat3 = nn.Sequential(nn.Conv2d(in_channels[1], out_channels, kernel_size=1),
                                  nn.GroupNorm(num_groups, out_channels))
        self.lat4 = nn.Sequential(nn.Conv2d(in_channels[2], out_channels, kernel_size=1),
                                  nn.GroupNorm(num_groups, out_channels))
        self.lat5 = nn.Sequential(nn.Conv2d(in_channels[3], out_channels, kernel_size=1),
                                  nn.GroupNorm(num_groups, out_channels))

        def smooth():
            return nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(num_groups, out_channels),
                nn.LeakyReLU(0.2, inplace=True))
        self.smooth2, self.smooth3 = smooth(), smooth()
        self.smooth4, self.smooth5 = smooth(), smooth()

        self.p6 = nn.MaxPool2d(kernel_size=1, stride=2, padding=0)

        self.dwt2_fuse = DWTEntropyFusion(in_channels[0], out_channels)
        self.dwt3_fuse = DWTEntropyFusion(in_channels[1], out_channels)
        self.dwt4_fuse = DWTEntropyFusion(in_channels[2], out_channels)

        self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]))
        self.register_buffer("lh", torch.tensor([[-0.5, -0.5], [0.5, 0.5]]))
        self.register_buffer("hl", torch.tensor([[-0.5, 0.5], [-0.5, 0.5]]))
        self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]))

    def dwt(self, x):
        B, C, H, W = x.shape
        if H % 2 != 0 or W % 2 != 0:
            x = F.pad(x, (0, W % 2, 0, H % 2))
        filters = torch.stack([self.ll, self.lh, self.hl, self.hh], dim=0).unsqueeze(1)
        filters = filters.repeat(C, 1, 1, 1)
        out = F.conv2d(x, filters, stride=2, groups=C)
        out = out.view(B, C, 4, out.shape[2], out.shape[3])
        return out[:, :, 1], out[:, :, 2], out[:, :, 3]

    def forward(self, C2, C3, C4, C5):
        P5_lat = self.lat5(C5)
        P4_lat = self.lat4(C4)
        P3_lat = self.lat3(C3)
        P2_lat = self.lat2(C2)

        P4 = P4_lat + F.interpolate(P5_lat, size=P4_lat.shape[2:], mode='nearest')
        P3 = P3_lat + F.interpolate(P4, size=P3_lat.shape[2:], mode='nearest')
        P2 = P2_lat + F.interpolate(P3, size=P2_lat.shape[2:], mode='nearest')

        LH2, HL2, HH2 = self.dwt(C2)
        LH3, HL3, HH3 = self.dwt(C3)
        LH4, HL4, HH4 = self.dwt(C4)

        F2 = F.interpolate(self.dwt2_fuse(LH2, HL2, HH2), size=P2.shape[2:], mode='bilinear', align_corners=False)
        F3 = F.interpolate(self.dwt3_fuse(LH3, HL3, HH3), size=P3.shape[2:], mode='bilinear', align_corners=False)
        F4 = F.interpolate(self.dwt4_fuse(LH4, HL4, HH4), size=P4.shape[2:], mode='bilinear', align_corners=False)

        P2 = self.smooth2(P2 + F2)
        P3 = self.smooth3(P3 + F3)
        P4 = self.smooth4(P4 + F4)
        P5 = self.smooth5(P5_lat)
        P6 = self.p6(P5)
        return P2, P3, P4, P5, P6

In [ ]:
# ============ 6. Backbones (from backbone.py) ============
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    mobilenet_v2, MobileNet_V2_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    densenet121, DenseNet121_Weights,
)


def _get_resnet_channels(resnet):
    def _layer_channels(layer):
        block = layer[-1]
        if hasattr(block, 'bn3'):
            return block.bn3.num_features
        return block.bn2.num_features
    return [_layer_channels(resnet.layer1), _layer_channels(resnet.layer2),
            _layer_channels(resnet.layer3), _layer_channels(resnet.layer4)]


class ResNetBackbone(nn.Module):
    def __init__(self, variant="resnet50", pretrained=False):
        super().__init__()
        weights_map = {
            "resnet18": ResNet18_Weights.IMAGENET1K_V1,
            "resnet34": ResNet34_Weights.IMAGENET1K_V1,
            "resnet50": ResNet50_Weights.IMAGENET1K_V1,
        }
        if variant not in weights_map:
            raise ValueError(f"Unsupported ResNet variant: {variant}")
        resnet_fn = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}[variant]
        weights = weights_map[variant] if pretrained else None
        resnet = resnet_fn(weights=weights)

        self.stage0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.stage1 = resnet.layer1
        self.stage2 = resnet.layer2
        self.stage3 = resnet.layer3
        self.stage4 = resnet.layer4
        self.out_channels = _get_resnet_channels(resnet)

    def forward(self, x):
        x = self.stage0(x)
        C2 = self.stage1(x)
        C3 = self.stage2(C2)
        C4 = self.stage3(C3)
        C5 = self.stage4(C4)
        return C2, C3, C4, C5


class MobileNetV2Backbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        model = mobilenet_v2(weights=weights)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(*model.features[3:7])
        self.stage2 = nn.Sequential(*model.features[7:14])
        self.stage3 = nn.Sequential(*model.features[14:])
        self.out_channels = [24, 32, 96, 1280]

    def forward(self, x):
        C2 = self.stage0(x)
        C3 = self.stage1(C2)
        C4 = self.stage2(C3)
        C5 = self.stage3(C4)
        return C2, C3, C4, C5


class EfficientNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = efficientnet_b0(weights=weights)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(model.features[3])
        self.stage2 = nn.Sequential(model.features[4])
        self.stage3 = nn.Sequential(*model.features[5:])
        self.out_channels = [24, 40, 80, 1280]

    def forward(self, x):
        C2 = self.stage0(x)
        C3 = self.stage1(C2)
        C4 = self.stage2(C3)
        C5 = self.stage3(C4)
        return C2, C3, C4, C5


class DenseNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = densenet121(weights=weights)
        self.stage0 = nn.Sequential(*model.features[:5])
        self.stage1 = nn.Sequential(model.features[5])
        self.stage2 = nn.Sequential(model.features[6], model.features[7])
        self.stage3 = nn.Sequential(*model.features[8:])
        self.out_channels = [256, 128, 256, 1024]

    def forward(self, x):
        C2 = self.stage0(x)
        C3 = self.stage1(C2)
        C4 = self.stage2(C3)
        C5 = self.stage3(C4)
        return C2, C3, C4, C5


def build_backbone(backbone_name="resnet50", pretrained=False):
    backbone_name = backbone_name.lower()
    if backbone_name == "resnet20":
        backbone_name = "resnet18"
    if backbone_name in {"resnet18", "resnet34", "resnet50"}:
        return ResNetBackbone(variant=backbone_name, pretrained=pretrained)
    if backbone_name == "mobilenet_v2":
        return MobileNetV2Backbone(pretrained=pretrained)
    if backbone_name == "efficientnet_b0":
        return EfficientNetBackbone(pretrained=pretrained)
    if backbone_name == "densenet121":
        return DenseNetBackbone(pretrained=pretrained)
    raise ValueError(f"Unsupported backbone: {backbone_name}")

In [ ]:
# ============ 7. Segmentation Model (from seg_head.py) ============
class SegModel(nn.Module):
    """Multi-scale 1/4 resolution summation with Instance Normalization."""
    def __init__(self, backbone_name="resnet50", pretrained=True, fpn_out_channels=256, num_classes=1):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = build_backbone(backbone_name, pretrained=pretrained)
        self.fpn = ModifiedFPN(in_channels=self.backbone.out_channels, out_channels=fpn_out_channels)

        self.decoder_conv = nn.Sequential(
            nn.Conv2d(fpn_out_channels, fpn_out_channels // 2, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fpn_out_channels // 2, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.final_upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, num_classes, kernel_size=3, padding=1),
        )

    def extract_features(self, x):
        C2, C3, C4, C5 = self.backbone(x)
        return self.fpn(C2, C3, C4, C5)

    def forward(self, x):
        P2, P3, P4, P5, P6 = self.extract_features(x)
        target_size = P2.shape[2:]
        u3 = F.interpolate(P3, size=target_size, mode="bilinear", align_corners=False)
        u4 = F.interpolate(P4, size=target_size, mode="bilinear", align_corners=False)
        u5 = F.interpolate(P5, size=target_size, mode="bilinear", align_corners=False)
        u6 = F.interpolate(P6, size=target_size, mode="bilinear", align_corners=False)
        fused = P2 + u3 + u4 + u5 + u6
        feat = self.decoder_conv(fused)
        return self.final_upsample(feat)

In [ ]:
# ============ 8. Training / evaluation helpers (from train.py) ============
def dice_score(preds, targets, smooth=1e-6):
    preds = preds.astype(np.float32)
    targets = targets.astype(np.float32)
    intersection = np.sum(preds * targets)
    return (2.0 * intersection + smooth) / (np.sum(preds) + np.sum(targets) + smooth)


def set_bn_eval(m):
    if "BatchNorm" in m.__class__.__name__:
        m.eval()


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    all_preds_bin, all_masks, dice_scores = [], [], []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = total_loss(preds, masks)
            running_loss += loss.item()

            probs = torch.sigmoid(preds)
            pred_bin = (probs > 0.5).float().cpu().numpy()
            mask_np = masks.cpu().numpy()

            dice_scores.append(dice_score(pred_bin, mask_np))
            all_preds_bin.append(pred_bin.astype(np.uint8).flatten())
            all_masks.append(mask_np.astype(np.uint8).flatten())

    all_masks = np.concatenate(all_masks)
    all_preds_bin = np.concatenate(all_preds_bin)

    return (running_loss / len(loader), np.mean(dice_scores),
            jaccard_score(all_masks, all_preds_bin, zero_division=0),
            accuracy_score(all_masks, all_preds_bin),
            precision_score(all_masks, all_preds_bin, zero_division=0),
            recall_score(all_masks, all_preds_bin, zero_division=0),
            f1_score(all_masks, all_preds_bin, zero_division=0))

In [ ]:
# ============ 9. Build loaders and model ============
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_files = list_image_files(cfg.train_img_dir)
train_files, val_files, _ = split_image_list(all_files, test_size=0.1, val_size=0.15, random_state=42)
print(f"Split: {len(train_files)} train / {len(val_files)} val")

train_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                          augment=cfg.augment, augment_factor=cfg.augment_factor,
                          augment_prob=cfg.augment_prob, rotation_range=cfg.rotation_range,
                          scale_range=cfg.scale_range, image_list=train_files)
val_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                        augment=False, image_list=val_files)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

model = SegModel(backbone_name=cfg.backbone, pretrained=cfg.pretrained,
                 num_classes=cfg.num_classes).to(device)

if cfg.pretrained_weights and os.path.exists(cfg.pretrained_weights):
    print(f"Loading UNITPathSSL pretrained checkpoint: {cfg.pretrained_weights}")
    checkpoint = torch.load(cfg.pretrained_weights, map_location=device)
    state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

In [ ]:
# ============ 10. Training loop ============
best_val_dice = 0.0
best_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_best.pth")
last_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_last.pth")
log_csv_path = os.path.join(cfg.save_dir, f"{cfg.backbone}_training_metrics.csv")

csv_headers = ["epoch", "train_loss", "val_loss", "val_dice", "val_iou", "val_accuracy",
               "val_precision", "val_recall", "val_f1", "lr", "epoch_time_sec"]
with open(log_csv_path, mode="w", newline="") as f:
    csv.writer(f).writerow(csv_headers)

print(f"Starting training: Backbone={cfg.backbone}, Epochs={cfg.epochs}, "
      f"BatchSize={cfg.batch_size}, LR={cfg.lr}")

epoch_durations = []
total_start_time = time.time()

for epoch in range(1, cfg.epochs + 1):
    epoch_start_time = time.time()
    model.train()
    if cfg.freeze_bn:
        model.apply(set_bn_eval)
    running_loss = 0.0
    optimizer.zero_grad()

    for i, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = total_loss(preds, masks) / cfg.accumulation_steps
        loss.backward()

        if (i + 1) % cfg.accumulation_steps == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * cfg.accumulation_steps

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    train_loss = running_loss / len(train_loader)
    val_loss, val_dice, val_iou, val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader, device)

    epoch_elapsed = time.time() - epoch_start_time
    epoch_durations.append(epoch_elapsed)

    with open(log_csv_path, mode="a", newline="") as f:
        csv.writer(f).writerow([
            epoch, f"{train_loss:.6f}", f"{val_loss:.6f}", f"{val_dice:.6f}",
            f"{val_iou:.6f}", f"{val_acc:.6f}", f"{val_prec:.6f}", f"{val_rec:.6f}",
            f"{val_f1:.6f}", f"{current_lr:.8f}", f"{epoch_elapsed:.2f}"])

    print(f"Epoch [{epoch:03d}/{cfg.epochs:03d}] ({epoch_elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Dice: {val_dice:.4f} | IoU: {val_iou:.4f} | Acc: {val_acc:.4f}")

    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_dice": val_dice},
               last_checkpoint)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_checkpoint)
        print(f"--> Saved Best Model Checkpoint (Val Dice: {best_val_dice:.4f})")

total_elapsed = time.time() - total_start_time
print(f"\n==================== Training Summary ====================")
print(f"Total Training Time:    {total_elapsed/60:.2f} mins")
print(f"Average Time / Epoch:   {sum(epoch_durations)/len(epoch_durations):.2f}s")
print(f"Best Validation Dice:   {best_val_dice:.4f}")
print(f"Best Checkpoint:        {best_checkpoint}")

In [ ]:
# ============ 11. Test set evaluation (from test.py) ============
test_files = list_image_files(cfg.test_img_dir)
test_ds = MoNuSegDataset(cfg.test_img_dir, cfg.test_mask_dir, target_size=cfg.target_size,
                         augment=False, image_list=test_files)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers)

test_model = SegModel(backbone_name=cfg.backbone, pretrained=False).to(device)
test_model.load_state_dict(torch.load(best_checkpoint, map_location=device))
test_model.eval()

all_preds, all_masks_t, all_probs, dice_list = [], [], [], []
running_loss = 0.0
print(f"Evaluating {len(test_files)} test images using checkpoint: {best_checkpoint}")

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = test_model(imgs)
        running_loss += total_loss(preds, masks).item()
        probs = torch.sigmoid(preds)
        pred_bin = (probs > 0.5).float().cpu().numpy()
        mask_np = masks.cpu().numpy()
        dice_list.append(dice_score(pred_bin, mask_np))
        all_probs.append(probs.cpu().numpy().flatten())
        all_preds.append(pred_bin.astype(np.uint8).flatten())
        all_masks_t.append(mask_np.astype(np.uint8).flatten())

all_masks_t = np.concatenate(all_masks_t)
all_preds = np.concatenate(all_preds)
all_probs = np.concatenate(all_probs)

print("\n================== TEST SET EVALUATION RESULTS ==================")
print(f"Mean Dice Score  : {np.mean(dice_list):.4f}")
print(f"Mean IoU (Jaccard): {jaccard_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"F1-Score         : {f1_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Precision        : {precision_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Recall           : {recall_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Accuracy         : {accuracy_score(all_masks_t, all_preds):.4f}")
print(f"AUC-ROC          : {roc_auc_score(all_masks_t, all_probs):.4f}")
print(f"Mean Loss        : {running_loss / len(test_loader):.4f}")
print("=================================================================")

In [ ]:
# ============ 12. Plots: training curves + qualitative panels ============
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(log_csv_path)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df["epoch"], df["train_loss"], label="Train Loss")
axes[0].plot(df["epoch"], df["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(df["epoch"], df["val_dice"], label="Val Dice")
axes[1].plot(df["epoch"], df["val_iou"], label="Val IoU")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score"); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# Qualitative panels: [Original | Ground Truth | Prediction]
n_show = min(4, len(test_ds))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
test_model.eval()
with torch.no_grad():
    for row in range(n_show):
        img_t, mask_t = test_ds[row]
        pred = torch.sigmoid(test_model(img_t.unsqueeze(0).to(device)))[0, 0].cpu().numpy()
        axes[row, 0].imshow(img_t.permute(1, 2, 0).numpy()); axes[row, 0].set_title("Original")
        axes[row, 1].imshow(mask_t[0].numpy(), cmap="gray"); axes[row, 1].set_title("Ground Truth")
        axes[row, 2].imshow(pred > 0.5, cmap="gray"); axes[row, 2].set_title("Prediction")
        for ax in axes[row]:
            ax.axis("off")
plt.tight_layout(); plt.show()

## Outputs
- Checkpoints + metrics CSV: `/kaggle/working/checkpoints/final/` (downloadable from the **Output** tab after saving a version).
- To resume with the UNITPathSSL checkpoint: upload `resnet50_stage2_pretrained.pth` as a private Kaggle dataset and set `Config.pretrained_weights` to its `/kaggle/input/...` path.